[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_10/notebook_10_1_shap_feature_importance.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 10.1: Feature Importance and SHAP Values

**Chapter 10: Interpretability and Explainability in Healthcare AI**

## Introduction: Making Black Box Models Transparent

A cardiologist reviews an AI's prediction:

```
Patient: 58-year-old male
AI Prediction: HIGH RISK of heart attack (0.83 probability)
Recommendation: Immediate catheterization
```

**The critical question**: *Why* does the AI think this patient is high risk?

Without an answer, the cardiologist faces an impossible choice:
- Trust blindly → risk unnecessary invasive procedure
- Ignore completely → risk missing life-threatening MI

**This notebook teaches you how to answer "why"** using:
1. **Permutation importance**: Which features matter most globally?
2. **SHAP values**: How does each feature contribute to this specific prediction?
3. **Clinical interpretation**: Translating math into actionable insights

## Learning Objectives

By the end of this notebook, you will:
- Understand the difference between global and local explanations
- Implement permutation importance for model validation
- Compute SHAP values for individual predictions
- Visualize feature importance with clinical annotations
- Detect spurious correlations that accuracy metrics miss
- Communicate AI predictions to clinicians effectively

---

## Part 1: Setup and Data

We'll use a cardiovascular disease prediction dataset.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from sklearn.inspection import permutation_importance
import shap
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("✓ Libraries imported")
print("\nThis notebook demonstrates how to explain AI predictions using SHAP.")
print("We'll make black box models transparent and clinically trustworthy.\n")

✓ Libraries imported

This notebook demonstrates how to explain AI predictions using SHAP.
We'll make black box models transparent and clinically trustworthy.



In [2]:
def create_cvd_dataset(n_samples=5000, include_spurious=False):
    """
    Create cardiovascular disease prediction dataset.

    Features (clinically relevant):
    - Age, gender, BMI
    - Blood pressure (systolic, diastolic)
    - Cholesterol (total, LDL, HDL)
    - Blood glucose
    - Smoking status, family history

    If include_spurious=True, adds:
    - Hospital ID (spurious correlation for demonstration)
    """
    data = []

    for i in range(n_samples):
        # Demographics
        age = np.random.normal(60, 15)
        age = np.clip(age, 30, 90)
        gender = np.random.choice(['M', 'F'])

        # Calculate CVD risk based on established clinical factors
        base_risk = 0.15

        # Age effect (exponential increase)
        if age > 65:
            base_risk *= 1.8
        elif age > 50:
            base_risk *= 1.3

        # Gender effect (males higher risk at younger ages)
        if gender == 'M' and age < 65:
            base_risk *= 1.4

        has_cvd = np.random.random() < base_risk

        # Clinical features (values depend on CVD status)
        # Distributions overlap substantially (as in real cohorts): each factor shifts risk only modestly
        if has_cvd:
            systolic_bp = np.random.normal(143, 18)
            diastolic_bp = np.random.normal(86, 11)
            total_chol = np.random.normal(220, 38)
            ldl_chol = np.random.normal(134, 32)
            hdl_chol = np.random.normal(48, 12)
            triglycerides = np.random.normal(155, 50)
            blood_glucose = np.random.normal(108, 25)
            bmi = np.random.normal(28, 5)
            smoking = np.random.random() < 0.30
            family_history = np.random.random() < 0.45
            exercise_hours = np.random.normal(1.9, 1.5)
        else:
            systolic_bp = np.random.normal(130, 16)
            diastolic_bp = np.random.normal(81, 10)
            total_chol = np.random.normal(205, 32)
            ldl_chol = np.random.normal(118, 28)
            hdl_chol = np.random.normal(53, 12)
            triglycerides = np.random.normal(140, 45)
            blood_glucose = np.random.normal(98, 22)
            bmi = np.random.normal(26.5, 4.5)
            smoking = np.random.random() < 0.18
            family_history = np.random.random() < 0.32
            exercise_hours = np.random.normal(2.8, 1.5)

        # Spurious feature (for demonstration of data leakage detection)
        if include_spurious:
            # Hospital A sees sicker patients (selection bias)
            if has_cvd:
                hospital_id = np.random.choice(['A', 'B', 'C'], p=[0.6, 0.2, 0.2])
            else:
                hospital_id = np.random.choice(['A', 'B', 'C'], p=[0.2, 0.4, 0.4])
        else:
            hospital_id = np.random.choice(['A', 'B', 'C'])

        data.append({
            'age': age,
            'gender': gender,
            'systolic_bp': systolic_bp,
            'diastolic_bp': diastolic_bp,
            'total_cholesterol': total_chol,
            'ldl_cholesterol': ldl_chol,
            'hdl_cholesterol': hdl_chol,
            'triglycerides': triglycerides,
            'blood_glucose': blood_glucose,
            'bmi': bmi,
            'smoking': int(smoking),
            'family_history': int(family_history),
            'exercise_hours_per_week': exercise_hours,
            'hospital_id': hospital_id,
            'cvd': int(has_cvd)
        })

    df = pd.DataFrame(data)

    # Clip to reasonable ranges
    df['age'] = df['age'].clip(30, 90)
    df['systolic_bp'] = df['systolic_bp'].clip(90, 200)
    df['diastolic_bp'] = df['diastolic_bp'].clip(50, 120)
    df['total_cholesterol'] = df['total_cholesterol'].clip(120, 400)
    df['ldl_cholesterol'] = df['ldl_cholesterol'].clip(50, 300)
    df['hdl_cholesterol'] = df['hdl_cholesterol'].clip(20, 100)
    df['triglycerides'] = df['triglycerides'].clip(50, 400)
    df['blood_glucose'] = df['blood_glucose'].clip(60, 250)
    df['bmi'] = df['bmi'].clip(15, 50)
    df['exercise_hours_per_week'] = df['exercise_hours_per_week'].clip(0, 10)

    return df

# Create dataset
df = create_cvd_dataset(n_samples=5000, include_spurious=False)

print("Dataset created!")
print(f"\nTotal samples: {len(df):,}")
print(f"CVD prevalence: {df['cvd'].mean():.1%}")
print("\nFirst few rows:")
df.head(10)

Dataset created!

Total samples: 5,000
CVD prevalence: 24.0%

First few rows:


,age,gender,systolic_bp,diastolic_bp,total_cholesterol,ldl_cholesterol,hdl_cholesterol,triglycerides,blood_glucose,bmi,smoking,family_history,exercise_hours_per_week,hospital_id,cvd
0,67.450712,M,127.787771,69.021936,273.533070,115.350612,41.854063,100.164634,88.931853,24.328215,0,0,3.046247,B,0
1,53.048735,F,122.548324,56.608942,224.310119,110.970769,51.033595,73.565164,130.713581,26.389952,0,0,3.333327,A,0
2,50.990420,F,125.332900,90.488429,255.587219,107.691214,57.506776,86.307880,89.000858,24.489663,0,0,5.086362,A,0
3,71.076999,F,132.741892,86.644642,205.816021,131.249411,60.910287,245.333585,121.561674,26.933874,0,1,3.428653,A,0
4,49.846170,F,154.010173,92.999356,185.544615,149.233363,63.643935,165.579351,122.926116,23.518324,0,0,1.732018,A,1
5,72.187887,M,151.699840,84.268452,202.404193,131.098253,61.833468,104.913415,79.434280,25.822598,0,0,1.351663,A,0
6,61.376412,F,107.223760,95.599756,306.914587,185.933456,57.884797,169.652962,130.415760,24.948390,0,1,1.425751,B,1
7,54.838792,M,142.012574,73.979469,194.514811,107.020972,35.437821,153.325412,103.743216,26.523011,0,1,2.448119,C,0
8,46.395192,M,116.922334,99.861859,210.586490,125.211411,52.106649,53.655295,97.416695,26.771036,1,1,6.494863,A,0
9,81.502732,M,147.052699,92.428228,229.061857,140.148895,42.087351,203.125744,67.159277,29.140857,1,0,6.085683,C,0


In [3]:
# Prepare data
# One-hot encode categorical variables
df_encoded = pd.get_dummies(df, columns=['gender', 'hospital_id'], drop_first=False)

feature_cols = [col for col in df_encoded.columns if col != 'cvd']
X = df_encoded[feature_cols].values
y = df_encoded['cvd'].values

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train):,}")
print(f"Test samples: {len(X_test):,}")
print(f"\nFeatures: {len(feature_cols)}")
print(f"Feature names: {feature_cols}")

Training samples: 4,000
Test samples: 1,000

Features: 17
Feature names: ['age', 'systolic_bp', 'diastolic_bp', 'total_cholesterol', 'ldl_cholesterol', 'hdl_cholesterol', 'triglycerides', 'blood_glucose', 'bmi', 'smoking', 'family_history', 'exercise_hours_per_week', 'gender_F', 'gender_M', 'hospital_id_A', 'hospital_id_B', 'hospital_id_C']


## Part 2: Train Model (Black Box)

We'll train a Random Forest—accurate but opaque.

In [4]:
# Train Random Forest
print("Training Random Forest model...\n")

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_split=20,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("✓ Model trained successfully")
print(f"\nTest Performance:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred):.1%}")
print(f"  AUC: {roc_auc_score(y_test, y_prob):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No CVD', 'CVD']))

# Performance in plain terms (computed): sensitivity/specificity at the default 0.5 threshold
tn, fp, fn, tp = ((y_pred == 0) & (y_test == 0)).sum(), ((y_pred == 1) & (y_test == 0)).sum(), \
                 ((y_pred == 0) & (y_test == 1)).sum(), ((y_pred == 1) & (y_test == 1)).sum()
sens, spec = tp / (tp + fn), tn / (tn + fp)
auc = roc_auc_score(y_test, y_prob)
print(f"\nAt the default 0.5 threshold: sensitivity {sens:.1%}, specificity {spec:.1%}")
quality = "good" if auc >= 0.85 else "moderate" if auc >= 0.75 else "weak"
print(f"Discrimination (AUC {auc:.3f}) is {quality}.")
if sens < 0.7:
    print(f"⚠️ Sensitivity is low at 0.5: {fn} of {tp + fn} CVD patients would be missed.")
    print("   0.5 is not a clinical choice: pick the threshold on validation data for the use case")
    print("   (e.g. screening favours high sensitivity).")

print("\n📦 Black Box Problem:")
print("Whatever its performance, we don't know WHY it makes predictions.")
print("Let's open the black box using explainability methods...\n")

Training Random Forest model...



✓ Model trained successfully

Test Performance:
  Accuracy: 83.1%
  AUC: 0.851

Classification Report:
              precision    recall  f1-score   support

      No CVD       0.84      0.95      0.90       760
         CVD       0.75      0.44      0.56       240

    accuracy                           0.83      1000
   macro avg       0.80      0.70      0.73      1000
weighted avg       0.82      0.83      0.81      1000


At the default 0.5 threshold: sensitivity 44.2%, specificity 95.4%
Discrimination (AUC 0.851) is good.
⚠️ Sensitivity is low at 0.5: 134 of 240 CVD patients would be missed.
   0.5 is not a clinical choice: pick the threshold on validation data for the use case
   (e.g. screening favours high sensitivity).

📦 Black Box Problem:
Whatever its performance, we don't know WHY it makes predictions.
Let's open the black box using explainability methods...



## Part 3: Permutation Importance (Global Explanation)

**Question**: Which features matter most across all predictions?

**Method**: Randomly shuffle each feature's values and measure performance drop.

In [5]:
print("Computing permutation importance...\n")
print("(This may take 30-60 seconds)\n")

# Compute permutation importance
perm_importance = permutation_importance(
    model, X_test, y_test,
    n_repeats=10,
    random_state=42,
    scoring='roc_auc'
)

# Create DataFrame for easier analysis
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': perm_importance.importances_mean,
    'std': perm_importance.importances_std
}).sort_values('importance', ascending=False)

print("✓ Permutation importance computed\n")
print("Top 10 Most Important Features:\n")
print(importance_df.head(10).to_string(index=False))

Computing permutation importance...

(This may take 30-60 seconds)



✓ Permutation importance computed

Top 10 Most Important Features:

                feature  importance      std
            systolic_bp    0.083467 0.010404
        ldl_cholesterol    0.038516 0.005538
exercise_hours_per_week    0.034775 0.003900
      total_cholesterol    0.019796 0.006640
          blood_glucose    0.019433 0.002234
                    bmi    0.013876 0.002746
        hdl_cholesterol    0.013722 0.003094
           diastolic_bp    0.012531 0.002258
          triglycerides    0.012311 0.001555
         family_history    0.006913 0.001144


In [6]:
# Visualize permutation importance
fig, ax = plt.subplots(figsize=(10, 8))

top_n = 15
top_features = importance_df.head(top_n)

colors = ['darkred' if imp > 0.01 else 'orange' if imp > 0.005 else 'gray'
          for imp in top_features['importance']]

bars = ax.barh(range(len(top_features)), top_features['importance'],
               xerr=top_features['std'], color=colors, edgecolor='black', linewidth=1.5)

ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'], fontsize=11)
ax.set_xlabel('Importance (Drop in AUC when permuted)', fontsize=12, fontweight='bold')
ax.set_title('Permutation Feature Importance\n(Higher = More Important)',
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, (bar, row) in enumerate(zip(bars, top_features.itertuples())):
    ax.text(row.importance + 0.001, i, f'{row.importance:.4f}',
            va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('permutation_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 CLINICAL INTERPRETATION:")
print("\nTop 5 drivers of CVD prediction:")
for rank, row in enumerate(importance_df.head(5).itertuples(), 1):
    feature = row.feature
    importance = row.importance

    # Clinical context
    if 'age' in feature.lower():
        context = "✓ Age is a known CVD risk factor"
    elif 'chol' in feature.lower():
        context = "✓ Cholesterol levels predict CVD"
    elif 'bp' in feature.lower() or 'blood_pressure' in feature.lower():
        context = "✓ Blood pressure is major CVD risk factor"
    elif 'glucose' in feature.lower():
        context = "✓ Glucose (diabetes marker) increases CVD risk"
    elif 'bmi' in feature.lower():
        context = "✓ Obesity is CVD risk factor"
    elif 'smoking' in feature.lower():
        context = "✓ Smoking is major modifiable risk factor"
    elif 'exercise' in feature.lower():
        j = feature_cols.index(feature)
        r_ex = np.corrcoef(np.asarray(X_test, dtype=float)[:, j], y_prob)[0, 1]   # direction learned by the model
        context = ("✓ Physical activity is a protective factor (more exercise → lower predicted risk)" if r_ex < 0
                   else "⚠️ Model links MORE exercise to HIGHER risk: implausible, investigate")
    elif 'hospital' in feature.lower():
        context = "⚠️ SPURIOUS: Hospital ID shouldn't matter (data leakage?)"
    else:
        context = ""

    print(f"  {rank}. {feature}: {importance:.4f}  {context}")

# Verdict computed from the ranking above (not assumed)
KNOWN_RISK_FACTORS = ('age', 'bp', 'chol', 'trigly', 'glucose', 'bmi', 'smok', 'family', 'exercise', 'gender')
top5 = list(importance_df.head(5)['feature'])
unexpected = [f for f in top5 if not any(k in f.lower() for k in KNOWN_RISK_FACTORS)]
if not unexpected:
    print("\n✅ All top-5 drivers are recognised CVD risk factors.")
    print("   That is necessary for trust, but not sufficient: a model can rank the right")
    print("   features and still be miscalibrated or rely on them in implausible directions.\n")
else:
    print(f"\n⚠️  {len(unexpected)} of the top-5 drivers are not recognised CVD risk factors: {', '.join(unexpected)}")
    print("   Investigate these before trusting the model (leakage? site effects? coding artefacts?).\n")


💡 CLINICAL INTERPRETATION:

Top 5 drivers of CVD prediction:
  1. systolic_bp: 0.0835  ✓ Blood pressure is major CVD risk factor
  2. ldl_cholesterol: 0.0385  ✓ Cholesterol levels predict CVD
  3. exercise_hours_per_week: 0.0348  ✓ Physical activity is a protective factor (more exercise → lower predicted risk)
  4. total_cholesterol: 0.0198  ✓ Cholesterol levels predict CVD
  5. blood_glucose: 0.0194  ✓ Glucose (diabetes marker) increases CVD risk

✅ All top-5 drivers are recognised CVD risk factors.
   That is necessary for trust, but not sufficient: a model can rank the right
   features and still be miscalibrated or rely on them in implausible directions.



## Part 4: SHAP Values (Local + Global Explanation)

SHAP (SHapley Additive exPlanations) provides:
- **Global**: Feature importance across all predictions
- **Local**: Contribution of each feature to a specific prediction

### Why SHAP?

1. **Theoretically grounded**: Based on game theory (Shapley values)
2. **Additive**: SHAP values sum to the difference between prediction and baseline
3. **Consistent**: If a feature becomes more important, its SHAP value doesn't decrease
4. **Local accuracy**: Faithfully represents model's reasoning for each prediction

In [7]:
# Initialize SHAP explainer
print("Initializing SHAP explainer...\n")
print("(TreeExplainer is fast for tree-based models like Random Forest)\n")

explainer = shap.TreeExplainer(model)

# Compute SHAP values for test set
print("Computing SHAP values for test set...")
print("(This may take 1-2 minutes for 1000 samples)\n")

shap_values = explainer.shap_values(X_test)

# For binary classification, shap_values is a list [class_0, class_1]
# We want class 1 (CVD positive)
if isinstance(shap_values, list):
    shap_values_cvd = shap_values[1]          # older SHAP: list [class_0, class_1]
elif shap_values.ndim == 3:
    shap_values_cvd = shap_values[:, :, 1]    # SHAP >= 0.45: (samples, features, classes)
else:
    shap_values_cvd = shap_values

print("✓ SHAP values computed successfully\n")
print(f"Shape: {shap_values_cvd.shape}")
print(f"(n_samples={shap_values_cvd.shape[0]}, n_features={shap_values_cvd.shape[1]})")

Initializing SHAP explainer...

(TreeExplainer is fast for tree-based models like Random Forest)

Computing SHAP values for test set...
(This may take 1-2 minutes for 1000 samples)



✓ SHAP values computed successfully

Shape: (1000, 17)
(n_samples=1000, n_features=17)


### SHAP Summary Plot (Global Feature Importance)

In [8]:
# Summary plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_cvd, X_test, feature_names=feature_cols, show=False)
plt.title('SHAP Summary Plot: Feature Importance Distribution',
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('shap_summary_plot.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 How to Read This Plot:")
print("\n  - Each dot is one patient")
print("  - Y-axis: Features (top = most important)")
print("  - X-axis: SHAP value (impact on prediction)")
print("    • Positive (right) = increases CVD risk")
print("    • Negative (left) = decreases CVD risk")
print("  - Color: Feature value (red = high, blue = low)")
# Read one feature from the plot, computed rather than assumed
X_arr = np.asarray(X_test, dtype=float)
mean_abs = np.abs(shap_values_cvd).mean(axis=0)
order = np.argsort(-mean_abs)
top = order[0]
r = np.corrcoef(X_arr[:, top], shap_values_cvd[:, top])[0, 1]
print(f"\n  Example: {feature_cols[top]} (the top-ranked feature)")
if r > 0.3:
    print("    • High values (red) sit on the right → higher values push predicted risk UP")
elif r < -0.3:
    print("    • High values (red) sit on the left → higher values push predicted risk DOWN")
else:
    print("    • No strong linear association: the effect may be non-linear (e.g. U-shaped)")
    print("      or depend on other features; look at a SHAP dependence plot to tell which")
print("    • Your turn: does that direction match clinical knowledge?")

age_rank = int(np.where(order == feature_cols.index('age'))[0][0]) + 1
print(f"\n  Age ranks {age_rank} of {len(feature_cols)} here, although age is a major real-world CVD risk factor.")
print("  This dataset is synthetic: the model can only learn the relationships the generator built in.")
print("  An explanation tells you what the MODEL uses, not what is clinically true.\n")


📊 How to Read This Plot:

  - Each dot is one patient
  - Y-axis: Features (top = most important)
  - X-axis: SHAP value (impact on prediction)
    • Positive (right) = increases CVD risk
    • Negative (left) = decreases CVD risk
  - Color: Feature value (red = high, blue = low)

  Example: systolic_bp (the top-ranked feature)
    • High values (red) sit on the right → higher values push predicted risk UP
    • Your turn: does that direction match clinical knowledge?

  Age ranks 11 of 17 here, although age is a major real-world CVD risk factor.
  This dataset is synthetic: the model can only learn the relationships the generator built in.
  An explanation tells you what the MODEL uses, not what is clinically true.



### SHAP Bar Plot (Average Absolute Impact)

In [9]:
# Bar plot (mean absolute SHAP values)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_cvd, X_test, feature_names=feature_cols,
                 plot_type='bar', show=False)
plt.title('SHAP Feature Importance (Mean |SHAP value|)',
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('shap_bar_plot.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Global Feature Importance (SHAP):")
print("\nThis shows the average magnitude of each feature's impact.")
print("Higher bars = more important for predictions overall.\n")


📊 Global Feature Importance (SHAP):

This shows the average magnitude of each feature's impact.
Higher bars = more important for predictions overall.



### SHAP Dependence Plots (How One Feature Shapes the Prediction)

The summary plot ranks features; a **dependence plot** shows *how* a single feature's value maps to its SHAP contribution
(x-axis = feature value, y-axis = SHAP value, colour = the feature SHAP picks as the strongest interaction).
A straight rising cloud means a roughly linear effect; a step or plateau reveals thresholds the model has learned.

In [10]:
# Dependence plots for the two top-ranked features (by mean |SHAP|)
X_test_df = pd.DataFrame(np.asarray(X_test, dtype=float), columns=feature_cols)
top_feats = [feature_cols[j] for j in order[:2]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, feat in zip(axes, top_feats):
    shap.dependence_plot(feat, shap_values_cvd, X_test_df, interaction_index='auto', ax=ax, show=False)
    ax.set_title(f'SHAP dependence: {feat}', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📈 What the dependence plots show (computed):")
for feat in top_feats:
    j = feature_cols.index(feat)
    vals, sv = X_test_df[feat].values, shap_values_cvd[:, j]
    lo_q, hi_q = np.quantile(vals, [0.25, 0.75])
    low_mean, high_mean = sv[vals <= lo_q].mean(), sv[vals >= hi_q].mean()
    rho = pd.Series(vals).corr(pd.Series(sv), method='spearman')
    direction = "raise" if high_mean > low_mean else "lower"
    print(f"  • {feat}: bottom quartile (≤{lo_q:.1f}) mean SHAP {low_mean:+.3f}, "
          f"top quartile (≥{hi_q:.1f}) {high_mean:+.3f} → higher values {direction} predicted risk")
    print(f"    Spearman ρ(value, SHAP) = {rho:.2f}"
          + ("  (near-monotonic)" if abs(rho) > 0.8 else "  (non-linear or interaction-dependent: look at the plot shape)"))



📈 What the dependence plots show (computed):
  • systolic_bp: bottom quartile (≤123.2) mean SHAP -0.076, top quartile (≥144.1) +0.150 → higher values raise predicted risk
    Spearman ρ(value, SHAP) = 0.95  (near-monotonic)
  • exercise_hours_per_week: bottom quartile (≤1.5) mean SHAP +0.096, top quartile (≥3.6) -0.057 → higher values lower predicted risk
    Spearman ρ(value, SHAP) = -0.95  (near-monotonic)


## Part 4b: Compare SHAP Explanations Across Model Types

Explanations describe a **model**, not the disease. If two well-performing models trained on the same data
disagree about which features matter, at least one explanation reflects modelling choices rather than biology.
We train two more models on the same split and explain each with the SHAP explainer suited to it:

| Model | SHAP explainer | Output scale |
|-------|----------------|--------------|
| Random Forest (above) | `TreeExplainer` | probability |
| XGBoost | `TreeExplainer` | log-odds |
| Logistic regression | `LinearExplainer` | log-odds |

Because the output scales differ, we compare **rankings and shares** of mean |SHAP|, not raw magnitudes.

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

X_tr_f = np.asarray(X_train, dtype=float)
X_te_f = np.asarray(X_test, dtype=float)

def positive_class_shap(sv):
    """Return (n_samples, n_features) SHAP values for the positive class, whatever SHAP returned."""
    if isinstance(sv, list):          # older SHAP: [class_0, class_1]
        sv = sv[1]
    sv = np.asarray(sv)
    if sv.ndim == 3:                  # SHAP >= 0.45: (samples, features, classes)
        sv = sv[:, :, 1]
    return sv                         # already 2-D (single log-odds output)

# XGBoost + TreeExplainer
xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                              eval_metric='logloss', random_state=42, n_jobs=1)
xgb_model.fit(X_tr_f, y_train)
sv_xgb = positive_class_shap(shap.TreeExplainer(xgb_model).shap_values(X_te_f))

# Logistic regression (standardised inputs) + LinearExplainer
scaler = StandardScaler().fit(X_tr_f)
lr_model = LogisticRegression(max_iter=2000).fit(scaler.transform(X_tr_f), y_train)
sv_lr = positive_class_shap(
    shap.LinearExplainer(lr_model, scaler.transform(X_tr_f)).shap_values(scaler.transform(X_te_f)))

models_shap = {
    'Random Forest': (shap_values_cvd, y_prob),
    'XGBoost': (sv_xgb, xgb_model.predict_proba(X_te_f)[:, 1]),
    'Logistic Regression': (sv_lr, lr_model.predict_proba(scaler.transform(X_te_f))[:, 1]),
}

share = {}
print(f"{'Model':<20} {'AUC':>6}   Top-5 features (share of total mean |SHAP|)")
print("-" * 100)
for name, (sv, prob) in models_shap.items():
    imp = np.abs(sv).mean(axis=0)
    share[name] = pd.Series(imp / imp.sum(), index=feature_cols)
    top5 = share[name].sort_values(ascending=False).head(5)
    print(f"{name:<20} {roc_auc_score(y_test, prob):>6.3f}   " +
          ", ".join(f"{f} ({s:.0%})" for f, s in top5.items()))
share_df = pd.DataFrame(share)


Model                   AUC   Top-5 features (share of total mean |SHAP|)
----------------------------------------------------------------------------------------------------
Random Forest         0.851   systolic_bp (19%), exercise_hours_per_week (13%), ldl_cholesterol (13%), total_cholesterol (11%), diastolic_bp (10%)
XGBoost               0.852   systolic_bp (16%), exercise_hours_per_week (12%), ldl_cholesterol (12%), diastolic_bp (9%), total_cholesterol (9%)
Logistic Regression   0.858   systolic_bp (16%), exercise_hours_per_week (12%), ldl_cholesterol (11%), diastolic_bp (9%), total_cholesterol (9%)


In [12]:
# Agreement between models (computed)
names = list(share_df.columns)
top5_sets = {n: set(share_df[n].sort_values(ascending=False).head(5).index) for n in names}
common = set.intersection(*top5_sets.values())

print("Pairwise agreement:")
min_overlap = 5
for a in range(len(names)):
    for b in range(a + 1, len(names)):
        n1, n2 = names[a], names[b]
        overlap = len(top5_sets[n1] & top5_sets[n2])
        min_overlap = min(min_overlap, overlap)
        rho = spearmanr(share_df[n1], share_df[n2]).correlation
        print(f"  {n1} vs {n2}: top-5 overlap {overlap}/5, Spearman ρ over all features = {rho:.2f}")

print(f"\nFeatures in the top 5 of ALL three models: {sorted(common) if common else 'none'}")
if min_overlap >= 4:
    print("✅ The models largely agree on what drives risk: the explanation is robust to the model choice.")
elif min_overlap >= 2:
    print("⚠️ Partial agreement: the strongest drivers are shared, but lower-ranked features shift with the model.")
    print("   Report the shared drivers with confidence; treat model-specific ones as artefacts until checked.")
else:
    print("❌ The models disagree on the main drivers: the explanations reflect modelling choices.")
    print("   Do not present any single model's SHAP ranking as 'the' clinical explanation.")
only = {n: sorted(top5_sets[n] - common) for n in names}
for n, fs in only.items():
    if fs:
        print(f"   Only in {n}'s top 5: {', '.join(fs)}")

# Side-by-side shares for the union of top features
union = share_df.max(axis=1).sort_values(ascending=False).head(8).index
share_df.loc[union].iloc[::-1].plot.barh(figsize=(10, 6), edgecolor='black')
plt.xlabel('Share of total mean |SHAP|', fontweight='bold')
plt.title('Same data, three models: do the explanations agree?', fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Why they can differ: logistic regression can only express linear (log-odds) effects, so it")
print("   spreads credit differently from tree models that learn thresholds and interactions.")
print("   Correlated features (e.g. total vs LDL cholesterol) can also swap credit between models.\n")


Pairwise agreement:
  Random Forest vs XGBoost: top-5 overlap 5/5, Spearman ρ over all features = 0.97
  Random Forest vs Logistic Regression: top-5 overlap 5/5, Spearman ρ over all features = 0.98
  XGBoost vs Logistic Regression: top-5 overlap 5/5, Spearman ρ over all features = 0.98

Features in the top 5 of ALL three models: ['diastolic_bp', 'exercise_hours_per_week', 'ldl_cholesterol', 'systolic_bp', 'total_cholesterol']
✅ The models largely agree on what drives risk: the explanation is robust to the model choice.

💡 Why they can differ: logistic regression can only express linear (log-odds) effects, so it
   spreads credit differently from tree models that learn thresholds and interactions.
   Correlated features (e.g. total vs LDL cholesterol) can also swap credit between models.



## Part 5: Local Explanations - Individual Patient

Now the real clinical value: explaining specific predictions.

In [13]:
# Select a high-risk patient
high_risk_idx = np.where(y_prob > 0.75)[0]

if len(high_risk_idx) > 0:
    patient_idx = high_risk_idx[0]
else:
    patient_idx = np.argmax(y_prob)

patient_features = X_test[patient_idx]
patient_prob = y_prob[patient_idx]
patient_true_label = y_test[patient_idx]

print("="*80)
print("HIGH-RISK PATIENT ANALYSIS")
print("="*80 + "\n")

print(f"Patient ID: {patient_idx}")
print(f"AI Prediction: {'HIGH RISK' if patient_prob > 0.5 else 'LOW RISK'} ({patient_prob:.1%} probability)")
print(f"True Label: {'CVD' if patient_true_label == 1 else 'No CVD'}")
print(f"\nPatient Features:")

for feat_name, feat_val in zip(feature_cols, patient_features):
    # Format feature values nicely
    if feat_val == 0 or feat_val == 1:
        # Binary feature
        display_val = 'Yes' if feat_val == 1 else 'No'
    else:
        # Continuous feature
        display_val = f"{feat_val:.1f}"

    print(f"  {feat_name}: {display_val}")

print("\n" + "="*80)

HIGH-RISK PATIENT ANALYSIS

Patient ID: 1
AI Prediction: HIGH RISK (78.0% probability)
True Label: CVD

Patient Features:
  age: 47.5
  systolic_bp: 178.4
  diastolic_bp: 90.8
  total_cholesterol: 231.5
  ldl_cholesterol: 129.5
  hdl_cholesterol: 63.9
  triglycerides: 74.8
  blood_glucose: 113.3
  bmi: 30.0
  smoking: No
  family_history: No
  exercise_hours_per_week: 1.3
  gender_F: No
  gender_M: Yes
  hospital_id_A: Yes
  hospital_id_B: No
  hospital_id_C: No



### SHAP Waterfall Plot (Step-by-Step Explanation)

In [14]:
# Check if expected_value is an array/list (multi-class) or scalar (single output)
base_val = explainer.expected_value
if hasattr(base_val, '__len__') and len(base_val) > 1:
    # If it has multiple classes, grab the one for CVD Risk (Index 1)
    base_val = base_val[1]

# Ensure it is a pure Python float (removes numpy array wrappers)
base_val = float(base_val)

# PREPARE EXPLANATION OBJECT
explanation = shap.Explanation(
    values=shap_values_cvd[patient_idx], # Contribution to Class 1
    base_values=base_val,                      # Scalar baseline risk
    data=X_test[patient_idx],                  # Feature values
    feature_names=feature_cols
)

# 3. PLOT
plt.figure(figsize=(10, 6))
shap.plots.waterfall(explanation, show=False)

plt.title(f'SHAP Waterfall: Patient {patient_idx} ({patient_prob:.1%} CVD Risk)',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### SHAP Force Plot (Alternative Visualization)

In [15]:
# 1. Robustly extract scalar base value for Class 1
base_val = explainer.expected_value
if hasattr(base_val, '__len__') and len(base_val) > 1:
    base_val = base_val[1] # Select Class 1 base value
base_val = float(base_val) # Ensure it's a float

# 2. Select SHAP values for Class 1 only
# Input shape was likely (features, 2) -> we want (features,)
shap_values_single = shap_values_cvd[patient_idx]

plt.figure(figsize=(14, 3))

# 3. Generate Plot
shap.force_plot(
    base_value=base_val,
    shap_values=shap_values_single,
    features=X_test[patient_idx],
    feature_names=feature_cols,
    matplotlib=True,
    show=False
)

plt.title(f'SHAP Force Plot: Patient {patient_idx}', fontsize=14, fontweight='bold', y=1.5)
plt.tight_layout()
plt.savefig(f'shap_force_patient_{patient_idx}.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Force Plot Interpretation:")
print(f"  - Base value: {base_val:.3f} (average model output)")
print("  - Red bars: Features pushing the prediction HIGHER")
print("  - Blue bars: Features pushing the prediction LOWER")
print(f"  - Final Output: {base_val + shap_values_single.sum():.3f}\n")


📊 Force Plot Interpretation:
  - Base value: 0.240 (average model output)
  - Red bars: Features pushing the prediction HIGHER
  - Blue bars: Features pushing the prediction LOWER
  - Final Output: 0.780



### Clinical Interpretation (Plain Language)

In [16]:
import numpy as np

# 1. Class 1 (CVD Risk) contributions, already selected when computing SHAP values
patient_shap = shap_values_cvd[patient_idx]

# 2. FIX: Robust baseline extraction (as discussed previously)
baseline = explainer.expected_value
if hasattr(baseline, '__len__') and len(baseline) > 1:
    baseline = baseline[1]
baseline = float(baseline)

# 3. Ensure we have the raw feature values for display
patient_features = X_test[patient_idx]

# Get top contributing features
# Now shap_val is a simple float, so sorting works!
shap_importance = [(feat_name, shap_val) for feat_name, shap_val in zip(feature_cols, patient_shap)]
shap_importance_sorted = sorted(shap_importance, key=lambda x: abs(x[1]), reverse=True)

print("\n" + "="*80)
print("CLINICAL EXPLANATION FOR PHYSICIAN")
print("="*80 + "\n")

print(f"Patient {patient_idx}: {'HIGH RISK' if patient_prob > 0.5 else 'LOW RISK'} of CVD ({patient_prob:.1%})\n")

# Note: If your SHAP values are Log-Odds (default), this baseline percentage might look small/odd
# unless you converted explainer output to probability space.
print(f"Baseline risk (average patient): {baseline:.1%}\n")

print("Top 5 factors INCREASING risk:\n")
positive_factors = [(f, s) for f, s in shap_importance_sorted if s > 0]

for i, (feat, shap_val) in enumerate(positive_factors[:5], 1):
    # Find original feature value
    feat_idx = feature_cols.index(feat)
    feat_val = patient_features[feat_idx]

    # Format value: Yes/No only for genuinely binary features (0 exercise hours is not "No")
    if set(np.unique(np.asarray(X_test, dtype=float)[:, feat_idx])) <= {0.0, 1.0}:
        val_str = 'Yes' if feat_val == 1 else 'No'
    else:
        val_str = f"{feat_val:.1f}"

    # Clinical context logic
    if 'age' in feat.lower():
        risk_change = f"Increases risk score by {shap_val:.3f}" # Changed % to raw score for safety
        note = "(Age is non-modifiable risk factor)"
    elif 'chol' in feat.lower():
        risk_change = f"Increases risk score by {shap_val:.3f}"
        note = "(Consider statin therapy if elevated)"
    elif 'bp' in feat.lower():
        risk_change = f"Increases risk score by {shap_val:.3f}"
        note = "(Consider antihypertensive if >140/90)"
    elif 'smoking' in feat.lower():
        risk_change = f"Increases risk score by {shap_val:.3f}"
        note = "(Smoking cessation counseling recommended)"
    else:
        risk_change = f"Increases risk score by {shap_val:.3f}"
        note = ""

    print(f"  {i}. {feat} = {val_str}")
    print(f"     {risk_change}  {note}")
    print()

print("Top 3 factors DECREASING risk:\n")
negative_factors = [(f, s) for f, s in shap_importance_sorted if s < 0]

for i, (feat, shap_val) in enumerate(negative_factors[:3], 1):
    feat_idx = feature_cols.index(feat)
    feat_val = patient_features[feat_idx]

    if set(np.unique(np.asarray(X_test, dtype=float)[:, feat_idx])) <= {0.0, 1.0}:
        val_str = 'Yes' if feat_val == 1 else 'No'
    else:
        val_str = f"{feat_val:.1f}"

    print(f"  {i}. {feat} = {val_str}")
    print(f"     Decreases risk score by {abs(shap_val):.3f}")
    print()

print("="*80)
print("\n💡 CLINICAL DECISION SUPPORT:")
print("\nThis explanation helps the clinician:")
print("  ✓ Verify the AI's reasoning aligns with clinical knowledge")
print("  ✓ Identify modifiable risk factors for intervention")
print("  ✓ Communicate risk to the patient in understandable terms")
print("  ✓ Decide whether to order additional tests or treatments\n")


CLINICAL EXPLANATION FOR PHYSICIAN

Patient 1: HIGH RISK of CVD (78.0%)

Baseline risk (average patient): 24.0%

Top 5 factors INCREASING risk:

  1. systolic_bp = 178.4
     Increases risk score by 0.488  (Consider antihypertensive if >140/90)

  2. exercise_hours_per_week = 1.3
     Increases risk score by 0.068  

  3. diastolic_bp = 90.8
     Increases risk score by 0.029  (Consider antihypertensive if >140/90)

  4. bmi = 30.0
     Increases risk score by 0.008  

  5. blood_glucose = 113.3
     Increases risk score by 0.004  

Top 3 factors DECREASING risk:

  1. hdl_cholesterol = 63.9
     Decreases risk score by 0.019

  2. age = 47.5
     Decreases risk score by 0.016

  3. ldl_cholesterol = 129.5
     Decreases risk score by 0.011


💡 CLINICAL DECISION SUPPORT:

This explanation helps the clinician:
  ✓ Verify the AI's reasoning aligns with clinical knowledge
  ✓ Identify modifiable risk factors for intervention
  ✓ Communicate risk to the patient in understandable terms
  ✓

## Part 6: Detecting Spurious Correlations

Let's see how explainability catches data leakage.

In [17]:
# Create dataset WITH spurious correlation
print("Creating new dataset with spurious correlation (hospital ID)...\n")

df_spurious = create_cvd_dataset(n_samples=5000, include_spurious=True)

# Prepare data
df_spurious_encoded = pd.get_dummies(df_spurious, columns=['gender', 'hospital_id'], drop_first=False)
feature_cols_spurious = [col for col in df_spurious_encoded.columns if col != 'cvd']
X_spurious = df_spurious_encoded[feature_cols_spurious].values
y_spurious = df_spurious_encoded['cvd'].values

X_train_sp, X_test_sp, y_train_sp, y_test_sp = train_test_split(
    X_spurious, y_spurious, test_size=0.2, random_state=42, stratify=y_spurious
)

# Train model
model_spurious = RandomForestClassifier(
    n_estimators=100, max_depth=12, min_samples_split=20, random_state=42, n_jobs=-1
)
model_spurious.fit(X_train_sp, y_train_sp)

y_prob_sp = model_spurious.predict_proba(X_test_sp)[:, 1]

print(f"Model with spurious feature:")
print(f"  AUC: {roc_auc_score(y_test_sp, y_prob_sp):.3f}")
print("\n⚠️  High accuracy doesn't guarantee the model is learning the right things!\n")

Creating new dataset with spurious correlation (hospital ID)...



Model with spurious feature:
  AUC: 0.876

⚠️  High accuracy doesn't guarantee the model is learning the right things!



In [18]:
# Compute SHAP for spurious model
print("Computing SHAP for spurious model...\n")

explainer_sp = shap.TreeExplainer(model_spurious)
shap_values_sp = explainer_sp.shap_values(X_test_sp)

if isinstance(shap_values_sp, list):
    shap_values_sp_cvd = shap_values_sp[1]          # older SHAP: list [class_0, class_1]
elif shap_values_sp.ndim == 3:
    shap_values_sp_cvd = shap_values_sp[:, :, 1]    # SHAP >= 0.45: (samples, features, classes)
else:
    shap_values_sp_cvd = shap_values_sp

# Summary plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_sp_cvd, X_test_sp, feature_names=feature_cols_spurious,
                 plot_type='bar', show=False)
plt.title('SHAP Feature Importance (WITH Spurious Feature)',
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('shap_spurious_model.png', dpi=150, bbox_inches='tight')
plt.show()

# Where does the shortcut actually rank? (computed, not assumed)
mean_abs_sp = np.abs(shap_values_sp_cvd).mean(axis=0)
rank_of = {feature_cols_spurious[j]: r + 1 for r, j in enumerate(np.argsort(-mean_abs_sp))}
hosp_ranks = sorted((rank_of[f], f) for f in feature_cols_spurious if f.startswith('hospital_id'))
best_rank, best_feat = hosp_ranks[0]
n_feat = len(feature_cols_spurious)
print("\nHospital ID ranks: " + ", ".join(f"{f} #{r}" for r, f in hosp_ranks) + f"  (of {n_feat} features)")

# Judge the hospital dummies together: several mid-ranked dummies can add up to a large share
hosp_idx = [j for j, f in enumerate(feature_cols_spurious) if f.startswith('hospital_id')]
hosp_share = mean_abs_sp[hosp_idx].sum() / mean_abs_sp.sum()
print(f"Combined hospital_id share of total mean |SHAP|: {hosp_share:.1%}")

if best_rank <= 5 or hosp_share >= 0.10:
    print(f"\n⚠️  ALERT: possible shortcut. Hospital identity carries {hosp_share:.1%} of the attribution.")
else:
    print(f"\n🔎 Hospital identity carries only {hosp_share:.1%} of the attribution (best dummy: {best_feat} #{best_rank}).")
    print("   Why so low? When the clinical features already separate the classes very well,")
    print("   the hospital signal adds little and is drowned out.")
    print("   A low rank does NOT mean the shortcut is harmless: check performance per hospital")
    print("   and on an external site before deployment.")
print("\nWhy a hospital-ID shortcut is problematic:")
print("  - Hospital ID has no causal relationship with CVD")
print("  - It's a proxy for patient selection bias (Hospital A sees sicker patients)")
print("  - The model may fail at new hospitals (test this with a held-out site, not by assumption)")
print("  - Model learns 'where patient was seen' instead of 'clinical risk factors'")
print("\n💡 Explainability can surface shortcuts, but only when they are strong enough to rank.")
if best_rank <= 5:
    print("   Exercise: weaken the selection bias in the data generator (hospital probabilities for CVD")
    print("   vs non-CVD patients) and re-run. At what strength does hospital_id drop out of the top 5?\n")
else:
    print("   Exercise: increase the hospital-outcome association in the data generator and re-run.")
    print("   At what strength does hospital_id reach the top 5?\n")

Computing SHAP for spurious model...




Hospital ID ranks: hospital_id_A #1, hospital_id_C #8, hospital_id_B #15  (of 17 features)
Combined hospital_id share of total mean |SHAP|: 25.8%

⚠️  ALERT: possible shortcut. Hospital identity carries 25.8% of the attribution.

Why a hospital-ID shortcut is problematic:
  - Hospital ID has no causal relationship with CVD
  - It's a proxy for patient selection bias (Hospital A sees sicker patients)
  - The model may fail at new hospitals (test this with a held-out site, not by assumption)
  - Model learns 'where patient was seen' instead of 'clinical risk factors'

💡 Explainability can surface shortcuts, but only when they are strong enough to rank.
   Exercise: weaken the selection bias in the data generator (hospital probabilities for CVD
   vs non-CVD patients) and re-run. At what strength does hospital_id drop out of the top 5?



### External Check: Does the Shortcut Survive a New Site?

SHAP shows *what* the model uses; an external test shows what it *costs*. We draw a fresh **synthetic** sample from a
site with **no selection bias** (hospital is unrelated to outcome) and compare the clean and the spurious model there.

In [19]:
# External check: a new sample in which hospital assignment is unrelated to CVD (no selection bias)
df_ext = create_cvd_dataset(n_samples=2000, include_spurious=False)
X_ext_df = pd.get_dummies(df_ext, columns=['gender', 'hospital_id'], drop_first=False)
y_ext = X_ext_df.pop('cvd').values

ext_auc_clean = roc_auc_score(y_ext, model.predict_proba(X_ext_df.reindex(columns=feature_cols, fill_value=0).values)[:, 1])
ext_auc_sp = roc_auc_score(y_ext, model_spurious.predict_proba(
    X_ext_df.reindex(columns=feature_cols_spurious, fill_value=0).values)[:, 1])
int_auc_sp = roc_auc_score(y_test_sp, y_prob_sp)

print(f"{'Model':<28}{'Internal test AUC':>18}{'New site AUC':>14}")
print(f"{'Clean model':<28}{roc_auc_score(y_test, y_prob):>18.3f}{ext_auc_clean:>14.3f}")
print(f"{'Model with hospital shortcut':<28}{int_auc_sp:>18.3f}{ext_auc_sp:>14.3f}")

drop = int_auc_sp - ext_auc_sp
if drop >= 0.03:
    print(f"\n⚠️ The shortcut model loses {drop:.3f} AUC at the new site; its internal AUC was inflated by")
    print("   'where the patient was seen'.")
else:
    print(f"\n🔎 The shortcut model loses only {drop:.3f} AUC at the new site: here the shortcut did little harm,")
    print("   but that must be shown with external data, never assumed.")
clean_drop = roc_auc_score(y_test, y_prob) - ext_auc_clean
print(f"\nClean model change: {-clean_drop:+.3f} AUC "
      + ("(within sampling noise of a 2,000-patient sample)" if abs(clean_drop) < 0.04 else "(larger than expected: investigate)"))


Model                        Internal test AUC  New site AUC
Clean model                              0.851         0.823
Model with hospital shortcut             0.876         0.778

⚠️ The shortcut model loses 0.098 AUC at the new site; its internal AUC was inflated by
   'where the patient was seen'.

Clean model change: -0.027 AUC (within sampling noise of a 2,000-patient sample)


## Summary and Key Takeaways

### What We Learned

1. **Permutation Importance (Global)**:
   - Identifies which features matter most across all predictions
   - Fast and model-agnostic
   - Useful for quick model validation

2. **SHAP Values (Global + Local)**:
   - Global: Summary plot shows feature importance and effect directions
   - Local: Waterfall/force plots explain individual predictions
   - Theoretically grounded (Shapley values from game theory)
   - Additive: SHAP values sum to the prediction minus the baseline (the model's average output)

3. **Clinical Applications**:
   - **Build trust**: Physicians can verify AI reasoning aligns with medical knowledge
   - **Detect errors**: Catch spurious correlations and data leakage
   - **Guide decisions**: Identify modifiable risk factors for intervention
   - **Communicate**: Explain predictions to patients in plain language

4. **Practical Workflow**:
   ```
   1. Train model → evaluate accuracy
   2. Compute permutation importance → check top features make clinical sense
   3. Compute SHAP values → generate global and local explanations
   4. Review explanations with clinicians → iterate if needed
   5. Deploy with explanation interface for real-time clinical use
   ```

### Best Practices

✅ **Always explain before deploying**: Don't deploy black boxes in healthcare  
✅ **Validate explanations**: Check that top features align with medical knowledge  
✅ **Use multiple methods**: SHAP + permutation importance provide complementary insights  
✅ **Make it actionable**: Focus explanations on modifiable risk factors  
✅ **Test for spurious correlations**: Use explainability to catch data leakage  
✅ **Iterate with clinicians**: User testing ensures explanations are understandable  

### Limitations

⚠️ **Association ≠ Causation**: High SHAP value doesn't mean changing the feature will change the outcome  
⚠️ **Computational cost**: SHAP can be slow for very large datasets (use TreeSHAP for tree models)  
⚠️ **Correlated features**: SHAP may distribute importance unpredictably between correlated features  
⚠️ **Post-hoc explanations**: Methods like SHAP explain what the model *did*, not necessarily what it *should* do  

---

## Exercises

1. **Explain David Chen**: Chapter 3's David is 58, male, total cholesterol 220 mg/dL, HDL 45 mg/dL, blood pressure 138/88 mmHg, a nonsmoker, and his father had an MI at 62 (`family_history = 1`). Build his row in `feature_cols` order (fill LDL, triglycerides, glucose, BMI and exercise with test-set medians; choose one hospital dummy), then get `model.predict_proba` and his SHAP values from `explainer.shap_values`. Which features drive his score? Age ranked only 11th in Part 4: does this model weigh him the way Framingham would?

2. **A screening threshold**: At the default 0.5 threshold the model misses 134 of 240 CVD patients (Part 2). Hold out a validation split from `X_train`, choose the threshold that gives about 80% sensitivity, and report test specificity. Does Patient 1's SHAP explanation change when the threshold changes? Hint: SHAP explains the score, not the decision.

3. **Two global rankings**: Compare the order in `importance_df` (permutation importance) with mean |SHAP| (`mean_abs`). Compute a Spearman correlation over all 17 features and list where the two methods disagree. Rerun `permutation_importance` with more `n_repeats`: are the disagreements within the reported `std`?

4. **Correlated features**: Drop `total_cholesterol` from the features (keep `ldl_cholesterol`), retrain the Random Forest and recompute SHAP. Does LDL absorb the removed feature's share? This is the "correlated features" limitation listed below, seen in your own numbers.

5. **How strong must a shortcut be? (deeper)**: In `create_cvd_dataset(include_spurious=True)`, the hospital is drawn with `p=[0.6, 0.2, 0.2]` for CVD and `p=[0.2, 0.4, 0.4]` for non-CVD patients. Move both towards `[1/3, 1/3, 1/3]` in steps. For each step, record the combined `hospital_id` share of mean |SHAP| (Part 6) and the AUC drop at the new site (external check). Is there a setting where SHAP no longer raises the alert but the new-site AUC still drops?

6. **Reflection: does the explanation change David's care?**: In Chapter 3, an ML model gave David 14.8% 10-year risk against Framingham's 12.4%, and the decision (a statin) was the same. If his SHAP waterfall showed blood pressure and low exercise as the main drivers, how would you explain that to him? What would you check before calling exercise a "modifiable lever" (association is not causation)?

---

### Next Steps

In the next notebooks, we'll explore:
- **Notebook 10.2**: LIME for local explanations (alternative to SHAP)
- **Notebook 10.3**: GradCAM for medical imaging (explaining CNNs)
- **Notebook 10.4**: Counterfactual explanations ("what if" scenarios)

---

**Remember**: In healthcare AI, explainability is not a luxury—it's a necessity for safety, trust, and clinical adoption.

*"A model you don't understand is a model you shouldn't trust with patient care."*